# Pandas 综合练习

数据集：`retail_data.csv` — 406条零售订单，预埋缺失值、重复值、脏数据类型和离群值。

**每题下方第一个代码格请你作答，第二个代码格是参考答案。**

## 题目1：读取与探索

1. 读取 `D:/dsml/data/retail_data.csv`，将 `Date` 列解析为日期时间类型。
2. 用 `info()` 和 `describe()` 查看数据结构。
3. 回答：哪些列有缺失值？`Revenue` 列的均值和中位数分别是多少？

In [ ]:
# ===== 你的作答 =====
import pandas as pd
import numpy as np
data = pd.read_csv(r'D:/dsml/data/retail_data.csv')
data['Date'] = pd.to_datetime(data['Date'], format='mixed')

In [ ]:
print(data.info())
print(data.describe())

In [ ]:
print(data.isna().sum())
print(data['Revenue'].mean(), data['Revenue'].median())

In [ ]:
# ===== 题目1 参考答案 =====
import pandas as pd
import numpy as np
df = pd.read_csv(r'D:/dsml/data/retail_data.csv', parse_dates=['Date'], date_format='mixed')
df.info()
miss = df.isnull().sum()
print('缺失值:', miss[miss > 0])
print(df.describe())
print(f"Revenue mean: {df['Revenue'].mean():.2f}, median: {df['Revenue'].median():.2f}")


## 题目2：缺失值处理

1. `Region` 缺失的行，统一填为 `'Unknown'`。
2. `Quantity` 缺失的行，按每个产品的众数填充（注意 Quantity 里有字符串 `N/A`）。
3. `Revenue` 缺失的行，用公式 `Quantity * UnitPrice * (1-Discount)` 重算并填充。

In [ ]:
# ===== 你的作答 =====
data['Region'] = data['Region'].fillna('Unknown')

In [ ]:
data0 = data.copy()
# 法一
P_mode = data.groupby('Product')['Quantity'].apply(lambda g: g.mode().iloc[0]) # 用 agg 替换 apply 也可
data['Quantity'] = data.apply(lambda g: P_mode[g['Product']] if pd.isna(g['Quantity']) else g['Quantity'], axis=1)
# 等价于 data['Quantity'] = data['Quantity'].fillna(data['Product'].map(P_mode))

# 法二
data0['P_mode'] = data0.groupby('Product')['Quantity'].transform(lambda g: g.mode().iloc[0])
mask = data0['Quantity'].isna()
data0.loc[mask, 'Quantity'] = data0.loc[mask, 'P_mode']
# 等价于 data0['Quantity'] = data0.apply(lambda g: g['P_mode'] if pd.isna(g['Quantity']) else g['Quantity'], axis=1)


In [ ]:
data['Revenue'] = data['Revenue'].fillna(data['Quantity']*data['UnitPrice']*(1-data['Discount']))

In [ ]:
# ===== 题目2 参考答案 =====
df['Region'] = df['Region'].fillna('Unknown')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
mode_map = df.groupby('Product')['Quantity'].agg(lambda x: x.mode().iloc[0])
df['Quantity'] = df['Quantity'].fillna(df['Product'].map(mode_map))
mask = df['Revenue'].isna()
df.loc[mask, 'Revenue'] = df.loc[mask, 'Quantity'] * df.loc[mask, 'UnitPrice'] * (1 - df.loc[mask, 'Discount'])
print('剩余缺失:', df.isnull().sum()[df.isnull().sum() > 0])


## 题目3：数据类型转换与清洗

1. `Category`、`Region`、`Customer` 转为 `category` 类型。
2. `Quantity` 转为 `int`。
3. 检查 `UnitPrice` 是否有 >50000 的离群值，有则用同产品中位数替换。

In [ ]:
# ===== 你的作答 =====
for col in ['Category', 'Region', 'Customer']:
    data[col] = data[col].astype('category')

In [ ]:
data['Quantity'] = data['Quantity'].astype('int')

In [ ]:
P_median = data.groupby('Product')['UnitPrice'].transform('median')
mask = data['UnitPrice']>50000
data.loc[mask, 'UnitPrice'] = P_median[mask]

In [ ]:
# ===== 题目3 参考答案 =====
for col in ['Category', 'Region', 'Customer']:
    df[col] = df[col].astype('category')
df['Quantity'] = df['Quantity'].astype('int')
outliers = df['UnitPrice'] > 50000
print(f'离群值: {outliers.sum()}')
if outliers.sum() > 0:
    median_p = df.groupby('Product')['UnitPrice'].transform('median')
    df.loc[outliers, 'UnitPrice'] = median_p[outliers]
print(df.dtypes)


## 题目4：去重

1. 检查完全重复的行数量。
2. 删除重复行，保留首次出现的记录。

In [ ]:
# ===== 你的作答 =====
print(data.duplicated().sum())

In [ ]:
data = data.drop_duplicates(keep='first')

In [ ]:
# ===== 题目4 参考答案 =====
print(f'重复行: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'去重后: {len(df)} 行')


## 题目5：数据分箱

1. 用 `pd.cut` 把 `UnitPrice` 分为4段，标签 `Budget`/`Standard`/`Premium`/`Luxury`。
2. 用 `pd.qcut` 把 `Revenue` 按人数等分为3段。
3. 统计每段订单数。

In [ ]:
# ===== 你的作答 =====
data['Price_Tier'] = pd.cut(data['UnitPrice'], bins=4, labels=['Budget','Standard','Premium','Luxury'])

In [ ]:
data['Revenue_Group'] = pd.qcut(data['Revenue'], q=3)

In [ ]:
print(data['Price_Tier'].value_counts())
print(data['Revenue_Group'].value_counts())

In [ ]:
# ===== 题目5 参考答案 =====
df['Price_Tier'] = pd.cut(df['UnitPrice'], bins=4, labels=['Budget','Standard','Premium','Luxury'])
df['Revenue_Group'] = pd.qcut(df['Revenue'], q=3, labels=['Low','Medium','High'])
print(df['Price_Tier'].value_counts())
print(df['Revenue_Group'].value_counts())


## 题目6：时间序列分析

1. 从 `Date` 提取月份、星期几（0=周一）、是否为周末。
2. `Date` 设为索引后按月汇总总收入和订单数。
3. 用7日滑动窗口计算收入的移动平均（先按天汇总再做 rolling）。
4. 回答收入最高月份。

In [ ]:
# ===== 你的作答 =====
data['Month'] = data['Date'].dt.month
data['Weekday'] = data['Date'].dt.weekday
data['Is_Weekend'] = data['Weekday'].isin([5,6])

In [ ]:
idx_Date = data.set_index('Date').sort_index()
idx_Date_Monthly = idx_Date.resample('ME').agg(Total_Revenue=('Revenue','sum'), Order_Count=('OrderID','count'))
print(idx_Date_Monthly.head(8))

In [ ]:
idx_Date_Daily = idx_Date['Revenue'].resample('D').sum().rolling(window=7, min_periods=1).mean()

In [ ]:
print(idx_Date_Monthly['Total_Revenue'].idxmax())

In [ ]:
# ===== 题目6 参考答案 =====
df['Month'] = df['Date'].dt.month
df['Weekday'] = df['Date'].dt.weekday
df['Is_Weekend'] = df['Weekday'].isin([5,6])
print(df[['Date','Month','Weekday','Is_Weekend']].head(8))

df_idx = df.set_index('Date').sort_index()
monthly = df_idx.resample('ME').agg(Total_Revenue=('Revenue','sum'), Order_Count=('OrderID','count'))
print(monthly)

daily = df_idx['Revenue'].resample('D').sum()
ma7 = daily.rolling(window=7, min_periods=1).mean()
print('7日均线最后10天:', ma7.tail(10).tolist())

best = monthly['Total_Revenue'].idxmax()
print(f'最高月份: {best.strftime("%Y-%m")}')


## 题目7：数据变形

1. 创建透视表：行=`Region`，列=`Category`，值=`Revenue` 求和。
2. 用 `melt` 还原回长表。

In [ ]:
# ===== 你的作答 =====
p_table = data.pivot_table(values='Revenue', index='Region', columns='Category', aggfunc='sum', observed=True)
# 如果元素一一对应就等价于 pivot，如果有不是一一对应的，元素就按aggfunc的形式操作聚合为一个
# observed 控制组合数，False：显示全部 18 种组合，实际没出现的填 NaN；True：只显示数据里真正出现过的组合
print(p_table)

In [ ]:
m_table = p_table.reset_index().melt(id_vars='Region', var_name='Category', value_name='Revenue')
print(m_table)

In [ ]:
# ===== 题目7 参考答案 =====
pivot = df.pivot_table(values='Revenue', index='Region', columns='Category', aggfunc='sum', observed=True)
print(pivot)
melted = pivot.reset_index().melt(id_vars='Region', var_name='Category', value_name='Total_Revenue')
print(melted.head(8))


## 题目8：分组聚合

1. 按 `Region` 和 `Category` 分组，用命名聚合算总收入和订单数。
2. 用 `transform` 给每行加上该 Region 的平均 Revenue。
3. 用 `filter` 保留总 Revenue >50000 的 Region。
4. 用 `apply` 找出每个 Region 内 Revenue 最高的 3 笔订单。

In [ ]:
# ===== 你的作答 =====
gp = data.groupby(['Region','Category'], observed=True).agg(
    Total_Revenue=('Revenue','sum'),
    Total_Orders=('OrderID','count')
)
print(gp)

In [ ]:
data['Avg_Revenue'] = data.groupby('Region', observed=True)['Revenue'].transform('mean')

In [ ]:
br = data.groupby('Region', observed=True).filter(lambda g: g['Revenue'].sum()>50000)
print(br['Region'].unique().tolist())

In [ ]:
R_top3 = data.groupby('Region', observed=True)[['OrderID','Product','Revenue']].apply(lambda g: g.nlargest(3, 'Revenue'))
print(R_top3)

In [ ]:
# ===== 题目8 参考答案 =====
rc = df.groupby(['Region','Category'], observed=True).agg(Total_Revenue=('Revenue','sum'), Order_Count=('OrderID','count'))
print(rc.head(6))

df['Region_Avg'] = df.groupby('Region')['Revenue'].transform('mean')
print(df[['Region','Revenue','Region_Avg']].head(6))

big = df.groupby('Region', observed=True).filter(lambda g: g['Revenue'].sum() > 50000)
print('大Region:', big['Region'].unique().tolist())

top3 = df.groupby('Region', observed=True)[['OrderID','Product','Revenue']].apply(lambda g: g.nlargest(3,'Revenue'))
print(top3)


## 题目9：导出

将清洗后的完整数据导出为 `D:/dsml/data/retail_data_clean.csv`，不带行索引，编码 `utf-8-sig`。

In [ ]:
# ===== 你的作答 =====
data.to_csv(r'D:/dsml/data/retail_data_clean.csv', index=False, encoding='utf-8-sig')

In [ ]:
# ===== 题目9 参考答案 =====
df.to_csv(r'D:/dsml/data/retail_data_clean.csv', index=False, encoding='utf-8-sig')
print(f'导出 {len(df)} 行，缺失值: {df.isnull().sum().sum()}')
